# 2.2 Custom agent state

Default agent state is mostly `messages`. You can **extend** `AgentState` with extra fields (here `favourite_colour`) that tools read and write.

That is different from **runtime context** (next notebook): state is conversation data that tools update; context is per-invoke config you pass in from outside.

This notebook:

1. Defines `CustomState`.
2. Writes to state with a tool that returns `Command(update=...)`.
3. Reads state from `runtime.state` in a second tool.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## State schema

Subclass `AgentState` and add fields. Pass `state_schema=CustomState` into `create_agent` so the graph knows the extra key.

In [ ]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

A tool that returns `Command(update={...})` patches graph state. Include a `ToolMessage` with `tool_call_id=runtime.tool_call_id` so the tool turn completes correctly.

`ToolRuntime` is injected by LangChain — the model does not fill it in.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
from langchain.messages import HumanMessage

# Model should call update_favourite_colour with "green"
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
from pprint import pprint

# Look for favourite_colour on the returned state dict
pprint(response)

You can also **seed** state on invoke: pass `favourite_colour` in the input dict. New `thread_id` so this is not mixed with the previous thread.

In [ ]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

## Read state

`runtime.state["favourite_colour"]` is the current graph state. Same `thread_id` as the write so the checkpointer still has the colour after the first turn.

In [ ]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

In [ ]:
# Same thread: read tool should return green
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)